# MiniCells Core Validation 001b — Generalization vs Residual Memorization

This notebook freshly reruns the frozen Core Validation 001 training code, then performs the preregistered cumulative Fourier-pair sweep for 001b. No training mechanism is changed.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json, shutil

BRANCH = 'codex/core-validation-001b-residual-memorization'
RESULT_BRANCH = 'kaggle/core-validation-001b-residual-memorization-results'
REPO = 'https://github.com/ArcheLabs/mini-cells.git'
ROOT = Path('/kaggle/working/mini-cells')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, str(ROOT)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('HEAD', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('TREE', subprocess.check_output(['git', 'rev-parse', 'HEAD^{tree}'], text=True).strip())


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_core_validation_001.py', 'tests/test_core_validation_001b.py'], check=True)
print('Core Validation 001 + 001b tests passed.')


In [ ]:
import torch
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu_count': torch.cuda.device_count(), 'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
assert torch.cuda.is_available(), 'Formal Core Validation 001b requires a CUDA GPU.'


## Fresh parent rerun

The next cell runs the existing Core Validation 001 runner unchanged and saves fresh early/late checkpoints under the 001b result directory. The parent oracle is skipped because 001b retrains one oracle with the full cumulative sweep.


In [ ]:
OUT = ROOT / 'results' / 'core-validation-001b-residual-memorization'
SOURCE = OUT / 'source-001'
if OUT.exists():
    shutil.rmtree(OUT)
SOURCE.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, 'scripts/run_core_validation_001.py',
    '--device', 'cuda', '--skip-oracle', '--out', str(SOURCE)
], check=True)
parent = json.loads((SOURCE / 'raw.json').read_text())
print(json.dumps(parent['decision'], indent=2, sort_keys=True))


## Core Validation 001b analysis

All 15 non-DC conjugate Fourier pairs are ranked by late embedding energy. The analysis evaluates cumulative exclusion and restriction for k=0..15 on early seen/unseen, late old/heldout, controls, and a freshly trained cumulative-replay oracle.


In [ ]:
subprocess.run([
    sys.executable, 'scripts/analyze_core_validation_001b.py',
    '--device', 'cuda', '--source', str(SOURCE), '--out', str(OUT)
], check=True)
subprocess.run([sys.executable, 'scripts/report_core_validation_001b.py', '--out', str(OUT)], check=True)


In [ ]:
decision = json.loads((OUT / 'decision.json').read_text())
print(json.dumps(decision, indent=2, sort_keys=True))
import pandas as pd
display(pd.read_csv(OUT / 'runs.csv'))


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUT / 'frequency-exclusion-trajectories.png')))
display(Image(filename=str(OUT / 'membership-gap-trajectories.png')))
display(Image(filename=str(OUT / 'oracle-exclusion-trajectory.png')))


## Publish

The final cell publishes curated formal 001b results to `kaggle/core-validation-001b-residual-memorization-results`. It expects the existing Kaggle secret `GITHUB_TOKEN` with Contents read/write permission.


In [ ]:
subprocess.run([sys.executable, 'scripts/publish_core_validation_001b.py', '--push'], check=True)
print(f'Published Core Validation 001b results to {RESULT_BRANCH}.')
